In [1]:
pip install numpy pandas scipy matplotlib seaborn statsmodels

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

# **Task 1: Chi-Square Goodness-of-Fit Test*

Task 1: Chi-Square Goodness-of-Fit Test
Scenario: A mobile-game studio expects that players choose among four character classes in equal proportions (25 % each). After a recent update, they collected the following counts from 400 new players:

| Class   | Warrior | Mage | Rogue | Healer |
|---------|---------|------|-------|--------|
| Count   | 120     | 85   | 110   | 85     |

1. Store the observed counts in a NumPy array.
2. Compute the expected counts under the null hypothesis of equal preference.
3. Run st.chisquare(observed, f_exp=expected).
4. Report the χ² statistic, degrees of freedom, and p-value.
5. At α = 0.05, state whether you reject or fail to reject H₀.
6. In a Markdown cell, answer: "Which class (if any) appears over- or under-represented, and by how much?"

In [7]:
import numpy as np
from scipy import stats as st

# 1. Observed counts
observed = np.array([120, 85, 110, 85])

# 2. Expected counts
total_players = 400
expected = np.array([total_players * 0.25] * 4) 

st.chisquare(f_obs=observed, f_exp=expected)

# 3. Chi-square test
chi2_stat, p_value = st.chisquare(f_obs=observed, f_exp=expected)

# 4. Statistic, Degrees of Freedom, p-value
df = len(observed) - 1

print(f"Chi-square Statistic: {chi2_stat:.2f}")
print(f"Degrees of Freedom: {df}")
print(f"P-value: {p_value:.4f}")

# 5. Whether reject or fail to reject H₀. (alpha = 0.05)
alpha = 0.05
if p_value < alpha:
    print("Result: Reject H0")
else:
    print("Result: Fail to reject H0")

Chi-square Statistic: 9.50
Degrees of Freedom: 3
P-value: 0.0233
Result: Reject H0


**Interpretation:** 

**Warrior** (+20) and **Rogue** (+10) are over-represented; 

**Mage** (−15) and **Healer** (−15) are under-represented.

**Warrior** contributes the most to χ² (4.00 out of 9.50 total).


# **Task 2: Chi-Square Test of Independence*

**Scenario:** A marketing team wants to know whether subscription tier (Free, Basic, Premium) is associated with churn status (Churned, Retained). They collected data from 600 customers:

|          | Free | Basic | Premium |
|----------|------|-------|---------|
| Churned  | 90   | 60    | 30      |
| Retained | 110  | 140   | 170     |

1. Create the contingency table as a 2 × 3 NumPy array or pandas DataFrame.
2. Run st.chi2_contingency(table) and unpack χ², p-value, degrees of freedom, and expected frequencies.
3. Display the expected-frequency table.
4. Report the χ² statistic, df, and p-value.
5. State your decision and a plain-language interpretation (e.g. "There is / is not evidence of an association between subscription tier and churn.").

In [8]:
import numpy as np
import pandas as pd
from scipy import stats as st

# 1. The contingency table as a 2 × 3 NumPy array.
# Rows : Churned, Retained
# Columns : Free, Basic, Premium
observed = np.array([
    [90, 60, 30],   # Churned (Gedənlər)
    [110, 140, 170] # Retained (Qalanlar)
])

# 2. Chi-square test
chi2_stat, p_value, dof, expected = st.chi2_contingency(observed)

# 3. Expected frequencies table
expected_df = pd.DataFrame(
    expected, 
    index=['Churned', 'Retained'], 
    columns=['Free', 'Basic', 'Premium']
)

print("--- Expected Frequencies Table ---")
print(expected_df)
print("\n--- Test Results ---")
print(f"Chi-square Statistic (χ²): {chi2_stat:.2f}")
print(f"Degrees of Freedom (df): {dof}")
print(f"P-value: {p_value:.11f}") 

--- Expected Frequencies Table ---
           Free  Basic  Premium
Churned    60.0   60.0     60.0
Retained  140.0  140.0    140.0

--- Test Results ---
Chi-square Statistic (χ²): 42.86
Degrees of Freedom (df): 2
P-value: 0.00000000049


**Decision:** 

$p < 0.05$ --->  So , $H_0$ rejected.

**Interpretation:**  

There is evidence of an association between subscription tier and churn.

# **Task 3: Compute Cramér's V*

Using the χ² statistic and the contingency table from Task 2:

1. Write a function:

def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.
    
    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
2. Compute Cramér's V for the Task 2 result.

In [10]:
import numpy as np

# 1. Write Cramér's V function
def cramers_v(chi2, n, min_dim):
    """
    Returns Cramér's V.

    Parameters
    ----------
    chi2 : float
        Chi-square statistic.
    n : int
        Total number of observations.
    min_dim : int
        min(rows, cols) - 1 of the contingency table.
    """
    return np.sqrt(chi2 / (n * min_dim))


n = 600
min_dim = min(2, 3) - 1  

v_score = cramers_v(chi2_stat, n, min_dim)

print(f"Cramér's V Score: {v_score:.4f}")

Cramér's V Score: 0.2673


3. Classify the effect size using Cohen's benchmarks for df* = min(rows, cols) − 1:
    - Small ≈ 0.10, Medium ≈ 0.30, Large ≈ 0.50 (for 2 × 3 tables, df* = 1)

**Cramér's V Score: 0.2673 ---> It means Medium effect**

4. In a Markdown cell, answer: "Is the relationship between tier and churn weak, moderate, or strong? What does this mean for the marketing team?"

##### **This represents a moderate relationship between a customer's subscription tier and their churn status.**

**What this means for the Marketing Team:**

 - **Tier-Driven Loyalty:** There is a clear association between the type of plan a customer chooses and their likelihood of staying with the service. The data suggests that as customers move from Free to Premium tiers, their retention rate increases significantly.

 - **High Risk in Free Tier:** The "Free" segment shows a higher-than-expected churn rate. This indicates that users on the free plan may not feel enough "stickiness" or value to remain long-term.

 - **Strategic Opportunity:** Since the relationship is moderate (not weak), marketing interventions can be highly effective. The team should focus on "Upselling" strategies—converting Free users to Basic or Premium tiers—as paid users demonstrate much higher loyalty and lower churn.

 - **Next Steps:** We recommend investigating the "onboarding" experience for Free users or offering limited-time discounts for tier upgrades to stabilize the customer base.